In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
import pandas as pd
import os
import sys
from pathlib import Path
import arviz as az

sys.path.append(str(Path.cwd().parent))

from _utils._utils_spline import eval_spline_basis_equispaced_numeric
from _run._run_generate_distortion_data import max_ratio_cd, max_ratio_cond, plot_max_ratio_cond, plot_max_ratio_cond_implementation

In [1]:
f = """
[
X    (‘bi_normal_5_100_1000', 0.05), X ()
X (‘bi_normal_5_100_1000', 0.1), X ()
X (‘bi_normal_5_100_1000', 0.5), o ()
X (‘bi_normal_5_100_1000', 1.0), o ()

X (‘bi_normal_5_100_2000', 0.05), X ()
X (‘bi_normal_5_100_2000', 0.1), X ()
X (‘bi_normal_5_100_2000', 0.5), O ()
X (‘bi_normal_5_100_2000', 1.0), o ()

X (‘bi_normal_5_10_1000', 0.05), O (3u, 7u, 28u, 34u)
X (‘bi_normal_5_10_1000', 0.1), O (7u, 37u)  
X (‘bi_normal_5_10_1000', 0.5), X ()
X (‘bi_normal_5_10_1000', 1.0), X ()

X (‘bi_normal_5_10_2000', 0.05), X
X (‘bi_normal_5_10_2000', 0.1), X
X (‘bi_normal_5_10_2000', 0.5), X
X (‘bi_normal_5_10_2000', 1.0), X

X ‘bi_normal_5_50_1000', 0.05), X ()
X (‘bi_normal_5_50_1000', 0.1), x ()
X (‘bi_normal_5_50_1000', 0.5), x ()
X (‘bi_normal_5_50_1000', 1.0), x ()

X (‘bi_normal_5_50_2000', 0.05), X ()
X (‘bi_normal_5_50_2000', 0.1), X ()
X (‘bi_normal_5_50_2000', 0.5), O ()
X (‘bi_normal_5_50_2000', 1.0), x ()

O (‘exponential_5_100_1000', 0.05), O (12u, 17u, 29u, 33u, 38u, 42u)
O ('exponential_5_100_1000', 0.1), O (1u, 15u, 29u, 38u)
O ('exponential_5_100_1000', 0.5), O (0u, 9u, 12u, 17u, 42u, 48u)
O ('exponential_5_100_1000', 1.0), O (0u, 4u, 14u, 18u, 31u, 35u)

O ('exponential_5_100_2000', 0.05), O (12u, 17u, 33u, 47u)
O ('exponential_5_100_2000', 0.1), O (40u, 42u, 46u)
O ('exponential_5_100_2000', 0.5), O (3u, 37o, 46u)
O ('exponential_5_100_2000', 1.0), O (2u)

O (‘exponential_5_10_1000', 0.05), O (7u, 10u, 32u, 36u, 48u)
O (‘exponential_5_10_1000', 0.1), O (7u, 48u)
O (‘exponential_5_10_1000', 0.5), O (14u)
O (‘exponential_5_10_1000', 1.0), o ()

X ('exponential_5_10_2000', 0.05), x
X ('exponential_5_10_2000', 0.1), x
X ('exponential_5_10_2000', 0.5), x
X ('exponential_5_10_2000', 1.0), x

O (‘exponential_5_50_1000', 0.05), O (1u, 5u, 10u, 12u, 15u, 22u, 26u)
O (‘exponential_5_50_1000', 0.1), O (1u, 6u, 7u, 10u, 11u, 12u, 22u, 49u)
O (‘exponential_5_50_1000', 0.5), O (0u, 3u, 17u, 33u, 35u)
O (‘exponential_5_50_1000', 1.0), O (0u, 3u, 38u)

O (‘exponential_5_50_2000', 0.05), O (5u)
O (‘exponential_5_50_2000', 0.1), O (5u, 38u)
O (‘exponential_5_50_2000', 0.5), O (25u, 32o, 37u, 38u)
O (‘exponential_5_50_2000', 1.0), o ()

x ('uni_normal_5_100_1000', 0.05), x ()
x ('uni_normal_5_100_1000', 0.1), x ()
X (‘uni_normal_5_100_1000', 0.5), x ()
X (‘uni_normal_5_100_1000', 1.0), O (7u, 27u)

o (‘uni_normal_5_100_2000', 0.05), x (48u)
o ('uni_normal_5_100_2000', 0.1), x ()
X (‘uni_normal_5_100_2000', 0.5), O ()
o (‘uni_normal_5_100_2000', 1.0), o ()

O (‘uni_normal_5_10_1000', 0.05), O (7u, 32u, 43u)
o (‘uni_normal_5_10_1000', 0.1), O ()
o (‘uni_normal_5_10_1000', 0.5), O (5u, 15u, 44u, 45u, 46u)
o (‘uni_normal_5_10_1000', 1.0), o (11u)

O (‘uni_normal_5_10_2000', 0.05), X ()
O (‘uni_normal_5_10_2000', 0.1), X ()
x ('uni_normal_5_10_2000', 0.5), X ()
x (‘uni_normal_5_10_2000', 1.0), X ()

X (‘uni_normal_5_50_1000', 0.05), o ()
X (‘uni_normal_5_50_1000', 0.1), o ()
X (‘uni_normal_5_50_1000', 0.5), O (7u, 38u, 43u)
X (‘uni_normal_5_50_1000', 1.0), O (7u, 22u, 38u)

o ('uni_normal_5_50_2000', 0.05), x ()
o ('uni_normal_5_50_2000', 0.1), o (38u)
o (‘uni_normal_5_50_2000', 0.5), O (38u)
x ('uni_normal_5_50_2000', 1.0), x ()

X ('uniform_5_100_1000', 0.05), X
X ('uniform_5_100_1000', 0.1), X
X ('uniform_5_100_1000', 0.5), X
X ('uniform_5_100_1000', 1.0), X

X ('uniform_5_100_2000', 0.05), X
X ('uniform_5_100_2000', 0.1), X
X ('uniform_5_100_2000', 0.5), X
X ('uniform_5_100_2000', 1.0), X

O (‘uniform_5_10_1000', 0.05), O ()
o ('uniform_5_10_1000', 0.1), O ()
X (‘uniform_5_10_1000', 0.5), O (28u, 30u, 31u)
X (‘uniform_5_10_1000', 1.0), o ()

O (‘uniform_5_10_2000', 0.05), X ()
O (‘uniform_5_10_2000', 0.1), X ()
X (‘uniform_5_10_2000', 0.5), X ()
X (‘uniform_5_10_2000', 1.0), X ()

X ('uniform_5_50_1000', 0.05), X
X ('uniform_5_50_1000', 0.1), X
X ('uniform_5_50_1000', 0.5), X
X ('uniform_5_50_1000', 1.0), X

X ('uniform_5_50_2000', 0.05), x
X ('uniform_5_50_2000', 0.1), x
X ('uniform_5_50_2000', 0.5), x
X ('uniform_5_50_2000', 1.0) x
]"""

In [2]:
import re

result = {}

In [5]:
for line in f.strip().splitlines():
    line = re.sub(r'\b[xXoO]\b(?=\s*\()', '', line).strip()
    line = line.replace("‘", "'").replace("’", "'")
    # Extract the key tuple
    m = re.match(r"\((.*?)\)\s*,\s*\((.*?)\)", line)
    if not m:
        continue

    key_str, values_str = m.groups()

    # Convert the key into a Python tuple
    key = eval(f"({key_str})")

    # Parse entries like 28u
    values = [
        (int(num), letter)
        for num, letter in re.findall(r"(\d+)\s*([A-Za-z])", values_str)
    ]

    result[key] = values

In [6]:
f_sigma_distortion = {}
for key, values in result.items():
    if len(values) > 0:
        f_sigma_distortion[key] = values

In [7]:
f_sigma_distortion

{('bi_normal_5_10_1000', 0.05): [(3, 'u'), (7, 'u'), (28, 'u'), (34, 'u')],
 ('bi_normal_5_10_1000', 0.1): [(7, 'u'), (37, 'u')],
 ('exponential_5_100_1000', 0.05): [(12, 'u'),
  (17, 'u'),
  (29, 'u'),
  (33, 'u'),
  (38, 'u'),
  (42, 'u')],
 ('exponential_5_100_1000', 0.1): [(1, 'u'), (15, 'u'), (29, 'u'), (38, 'u')],
 ('exponential_5_100_1000', 0.5): [(0, 'u'),
  (9, 'u'),
  (12, 'u'),
  (17, 'u'),
  (42, 'u'),
  (48, 'u')],
 ('exponential_5_100_1000', 1.0): [(0, 'u'),
  (4, 'u'),
  (14, 'u'),
  (18, 'u'),
  (31, 'u'),
  (35, 'u')],
 ('exponential_5_100_2000', 0.05): [(12, 'u'),
  (17, 'u'),
  (33, 'u'),
  (47, 'u')],
 ('exponential_5_100_2000', 0.1): [(40, 'u'), (42, 'u'), (46, 'u')],
 ('exponential_5_100_2000', 0.5): [(3, 'u'), (37, 'o'), (46, 'u')],
 ('exponential_5_100_2000', 1.0): [(2, 'u')],
 ('exponential_5_10_1000', 0.05): [(7, 'u'),
  (10, 'u'),
  (32, 'u'),
  (36, 'u'),
  (48, 'u')],
 ('exponential_5_10_1000', 0.1): [(7, 'u'), (48, 'u')],
 ('exponential_5_10_1000', 0.5): [

In [27]:
tau_vals = [4, 5, 6, 7, 8]
comparison_list = [('ortho_conditioning', 'svd')]
penalised = False
builder = 'ortho_diag'

In [12]:
os.listdir()

['shrinkage_0.ipynb',
 'dengue_example.ipynb',
 'distortion_examples.html',
 'distortion_examples.ipynb',
 'prior.ipynb',
 'cherry_examples.ipynb',
 'gold_tau.ipynb']

In [ ]:
distortion_report_name = "Q_distortion"
distortion_report_folder = '../_distortion_reports'
distortion_report_path = os.path.join(distortion_report_folder, distortion_report_name)
if not os.path.exists(distortion_report_folder):
    os.makedirs(distortion_report_folder)
if not os.path.exists(distortion_report_path):
    os.makedirs(distortion_report_path)

In [28]:
def run_folder_name(tau):
    return f'run_Qtau{tau}_a1b1k5_dist_dataQ_5_000'

In [ ]:
for i1, i2 in comparison_list:
    for f_sigma, distortion_replications in f_sigma_distortion.items():
        for d_r, quality in distortion_replications:
            f = f_sigma[0]
            sigma = f_sigma[1]
            report_path = os.path.join(distortion_report_path, f"{quality}_({i1}_{i2})_{f}_{sigma}_{d_r}.html")

            html_parts = [f"<html><head><title>Distortion Report: {i1} vs {i2}</title>",
                                    "<style>",
                                    "body { font-family: Arial; font-size: 10px; line-height: 1.2; margin: 8px; text-align:center; }",
                                    "h1, h2 { margin: 4px 0 8px 0; font-weight: normal; }",
                                    "table { border-collapse: collapse; font-size: 15px; margin: 0 auto 12px auto; width: auto; }",
                                    "table th, table td { border: 1px solid #aaa; padding: 4px 6px; text-align: center; }",
                                    "img { max-width: 80%; margin: 8px auto; display: block; }",
                                    "</style></head><body>",
                                    f"<h1>Comparison Report: {i1} vs {i2}</h1>",
                                    f"<h1>{quality}_({i1}_{i2})_{f_sigma[0]}_{f_sigma[1]}_{d_r}</h1>"]

            idata_file_1 = f"idata_{f}_{sigma}_{i1}_{penalised}_{d_r}({builder}).nc"
            idata_file_2 = f"idata_{f}_{sigma}_{i2}_{penalised}_{d_r}({builder}).nc"
            for tau in tau_vals:
                idatas_path = os.path.join('../', f'run_{run_folder_name(tau)}')
                idata_path_1 = os.path.join(idatas_path, idata_file_1)
                idata_path_2 = os.path.join(idatas_path, idata_file_2)

                idata1 = az.from_netcdf(idata_path_1)
                idata2 = az.from_netcdf(idata_path_2)